# Retail Black Friday Sales Data Engineering Project

## Project Overview

This project demonstrates an end-to-end data engineering and analytics workflow using Python, pandas, Google Colab, and a retail Black Friday sales dataset.

The project starts from raw transaction data and creates clean, business-ready analytical outputs such as KPI tables, customer summaries, category reports, RFM analysis, and suspicious transaction reports.

## Dataset

Dataset file used:

`retail_black_friday_sales_100k.csv`

Source:

Kaggle — Black Friday Sales Dataset  
https://www.kaggle.com/datasets/noopurbhatt/retail-black-friday-sales-dataset?select=retail_black_friday_sales_100k.csv

## Dataset Columns

- transaction_id
- customer_id
- age_group
- gender
- city
- customer_segment
- product_id
- product_category
- original_price
- discount_pct
- final_price
- quantity
- purchase_amount
- payment_method
- purchase_date
- purchase_hour
- is_weekend
- is_black_friday

## Skills Covered

- Data ingestion
- Schema inspection
- Data quality checks
- Data cleaning
- Date transformation
- Feature engineering
- GroupBy aggregations
- KPI generation
- Customer summary table creation
- RFM analysis
- Suspicious transaction detection
- Exporting analytics-ready CSV files

## 1. Load Dataset

Upload `retail_black_friday_sales_100k.csv` into the Colab session before running this notebook.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("retail_black_friday_sales_100k.csv")

df.head()

## 2. Basic Data Inspection

This step checks row count, column count, schema, data types, null values, and duplicate rows.

In [ ]:
print("Rows and Columns:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
df.info()

In [ ]:
df.head()

In [ ]:
null_summary = df.isnull().sum().reset_index()
null_summary.columns = ["column_name", "null_count"]
null_summary

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_count

## 3. Data Cleaning

Remove duplicate rows and convert `purchase_date` into proper datetime format.

In [ ]:
df = df.drop_duplicates()

df["purchase_date"] = pd.to_datetime(
    df["purchase_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df.info()

## 4. Feature Engineering

Create useful date and business features.

In [ ]:
df["purchase_year"] = df["purchase_date"].dt.year
df["purchase_month"] = df["purchase_date"].dt.month
df["purchase_month_name"] = df["purchase_date"].dt.month_name()
df["purchase_day"] = df["purchase_date"].dt.day
df["purchase_weekday"] = df["purchase_date"].dt.day_name()

df["discount_amount"] = df["original_price"] - df["final_price"]

df["order_value_bucket"] = pd.cut(
    df["purchase_amount"],
    bins=[0, 100, 250, 500, 1000, float("inf")],
    labels=["0-100", "100-250", "250-500", "500-1000", "1000+"]
)

df.head()

# Business Analysis

## 5. Top Selling Product Categories

Business question:

Which product categories generate the highest revenue?

In [ ]:
top_categories = (
    df.groupby("product_category")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_quantity=("quantity", "sum"),
        total_transactions=("transaction_id", "nunique")
    )
    .reset_index()
    .sort_values(by="total_revenue", ascending=False)
)

top_categories

## 6. Revenue by City

Business question:

Which cities generate the highest sales?

In [ ]:
city_revenue = (
    df.groupby("city")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values(by="total_revenue", ascending=False)
)

city_revenue

## 7. Payment Method Analysis

Business question:

Which payment methods are most used and which generate the highest revenue?

In [ ]:
payment_analysis = (
    df.groupby("payment_method")
    .agg(
        total_transactions=("transaction_id", "nunique"),
        total_revenue=("purchase_amount", "sum"),
        avg_order_value=("purchase_amount", "mean")
    )
    .reset_index()
    .sort_values(by="total_transactions", ascending=False)
)

payment_analysis["transaction_percentage"] = (
    payment_analysis["total_transactions"] /
    payment_analysis["total_transactions"].sum()
) * 100

payment_analysis

## 8. Weekend vs Weekday Sales

Business question:

Do customers spend more on weekends or weekdays?

In [ ]:
weekend_sales = (
    df.groupby("is_weekend")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        avg_order_value=("purchase_amount", "mean")
    )
    .reset_index()
)

weekend_sales["day_type"] = weekend_sales["is_weekend"].map({
    0: "Weekday",
    1: "Weekend"
})

weekend_sales = weekend_sales[[
    "day_type",
    "total_revenue",
    "total_transactions",
    "avg_order_value"
]]

weekend_sales

## 9. Customer Segment Analysis

Business question:

Which customer segment contributes the most revenue?

In [ ]:
segment_analysis = (
    df.groupby("customer_segment")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        avg_order_value=("purchase_amount", "mean")
    )
    .reset_index()
    .sort_values(by="total_revenue", ascending=False)
)

segment_analysis

## 10. Discount Impact Analysis

Business question:

Do higher discounts increase revenue or order value?

In [ ]:
df["discount_bucket"] = pd.cut(
    df["discount_pct"],
    bins=[-1, 10, 20, 30, 40, 50, 100],
    labels=["0-10", "10-20", "20-30", "30-40", "40-50", "50+"]
)

discount_analysis = (
    df.groupby("discount_bucket")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        avg_order_value=("purchase_amount", "mean"),
        avg_quantity=("quantity", "mean")
    )
    .reset_index()
)

discount_analysis

## 11. Hourly Sales Trend

Business question:

At what hour do customers purchase the most?

In [ ]:
hourly_sales = (
    df.groupby("purchase_hour")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        avg_order_value=("purchase_amount", "mean")
    )
    .reset_index()
    .sort_values(by="purchase_hour")
)

hourly_sales

## 12. Age Group and Category Analysis

Business question:

Which age groups spend the most by product category?

In [ ]:
age_category_sales = (
    df.groupby(["age_group", "product_category"])
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique")
    )
    .reset_index()
    .sort_values(by="total_revenue", ascending=False)
)

age_category_sales.head(20)

## 13. Black Friday vs Normal Day Sales

Business question:

How much revenue comes from Black Friday compared to normal days?

In [ ]:
black_friday_analysis = (
    df.groupby("is_black_friday")
    .agg(
        total_revenue=("purchase_amount", "sum"),
        total_transactions=("transaction_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        avg_order_value=("purchase_amount", "mean")
    )
    .reset_index()
)

black_friday_analysis["sale_type"] = black_friday_analysis["is_black_friday"].map({
    0: "Normal Day",
    1: "Black Friday"
})

black_friday_analysis = black_friday_analysis[[
    "sale_type",
    "total_revenue",
    "total_transactions",
    "unique_customers",
    "avg_order_value"
]]

black_friday_analysis

## 14. KPI Table

Create a one-row business summary table.

In [ ]:
kpi_table = pd.DataFrame({
    "metric": [
        "total_revenue",
        "total_transactions",
        "unique_customers",
        "unique_products",
        "avg_order_value",
        "total_quantity",
        "avg_discount_pct"
    ],
    "value": [
        df["purchase_amount"].sum(),
        df["transaction_id"].nunique(),
        df["customer_id"].nunique(),
        df["product_id"].nunique(),
        df["purchase_amount"].mean(),
        df["quantity"].sum(),
        df["discount_pct"].mean()
    ]
})

kpi_table

## 15. Customer Summary Table

Create one row per customer.

This is similar to creating a customer-level data mart.

In [ ]:
customer_summary = (
    df.groupby("customer_id")
    .agg(
        total_transactions=("transaction_id", "nunique"),
        total_spend=("purchase_amount", "sum"),
        avg_order_value=("purchase_amount", "mean"),
        total_quantity=("quantity", "sum"),
        first_purchase_date=("purchase_date", "min"),
        last_purchase_date=("purchase_date", "max"),
        preferred_city=("city", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        preferred_category=("product_category", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        preferred_payment_method=("payment_method", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )
    .reset_index()
    .sort_values(by="total_spend", ascending=False)
)

customer_summary.head(10)

## 16. RFM Analysis

RFM means:

- Recency: how recently a customer purchased
- Frequency: how many times a customer purchased
- Monetary: how much a customer spent

This is a common customer analytics technique used in retail, marketing, and CRM.

In [ ]:
reference_date = df["purchase_date"].max()

rfm = (
    df.groupby("customer_id")
    .agg(
        recency=("purchase_date", lambda x: (reference_date - x.max()).days),
        frequency=("transaction_id", "nunique"),
        monetary=("purchase_amount", "sum")
    )
    .reset_index()
)

rfm.head()

In [ ]:
rfm["recency_score"] = pd.qcut(
    rfm["recency"],
    q=4,
    labels=[4, 3, 2, 1],
    duplicates="drop"
)

rfm["frequency_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
    duplicates="drop"
)

rfm["monetary_score"] = pd.qcut(
    rfm["monetary"],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates="drop"
)

rfm["rfm_score"] = (
    rfm["recency_score"].astype(str)
    + rfm["frequency_score"].astype(str)
    + rfm["monetary_score"].astype(str)
)

rfm.head()

In [ ]:
def assign_rfm_segment(row):
    if row["rfm_score"] == "444":
        return "Best Customers"
    elif row["recency_score"] >= 3 and row["frequency_score"] >= 3:
        return "Loyal Customers"
    elif row["recency_score"] <= 2 and row["frequency_score"] >= 3:
        return "At Risk"
    elif row["recency_score"] >= 3 and row["frequency_score"] <= 2:
        return "New / Recent Customers"
    else:
        return "Others"

rfm["rfm_segment"] = rfm.apply(assign_rfm_segment, axis=1)

rfm.head()

## 17. Suspicious Transaction Detection

Rule-based suspicious transaction detection.

Examples:

- very high discount
- very high quantity
- very high purchase amount
- late-night purchase

In [ ]:
purchase_amount_99 = df["purchase_amount"].quantile(0.99)
quantity_99 = df["quantity"].quantile(0.99)

suspicious_transactions = df[
    (df["discount_pct"] >= 40) |
    (df["quantity"] > quantity_99) |
    (df["purchase_amount"] > purchase_amount_99) |
    (df["purchase_hour"] <= 4)
].copy()

suspicious_transactions["suspicious_reason"] = ""

suspicious_transactions.loc[
    suspicious_transactions["discount_pct"] >= 40,
    "suspicious_reason"
] += "High Discount; "

suspicious_transactions.loc[
    suspicious_transactions["quantity"] > quantity_99,
    "suspicious_reason"
] += "High Quantity; "

suspicious_transactions.loc[
    suspicious_transactions["purchase_amount"] > purchase_amount_99,
    "suspicious_reason"
] += "High Purchase Amount; "

suspicious_transactions.loc[
    suspicious_transactions["purchase_hour"] <= 4,
    "suspicious_reason"
] += "Late Night Purchase; "

suspicious_transactions.head()

## 18. Export Final Outputs

These files are analytics-ready outputs that can be used for reporting or uploaded to GitHub.

In [ ]:
top_categories.to_csv("top_categories.csv", index=False)
city_revenue.to_csv("city_revenue.csv", index=False)
payment_analysis.to_csv("payment_analysis.csv", index=False)
weekend_sales.to_csv("weekend_sales.csv", index=False)
segment_analysis.to_csv("segment_analysis.csv", index=False)
discount_analysis.to_csv("discount_analysis.csv", index=False)
hourly_sales.to_csv("hourly_sales.csv", index=False)
age_category_sales.to_csv("age_category_sales.csv", index=False)
black_friday_analysis.to_csv("black_friday_analysis.csv", index=False)
kpi_table.to_csv("kpi_table.csv", index=False)
customer_summary.to_csv("customer_summary.csv", index=False)
rfm.to_csv("rfm_analysis.csv", index=False)
suspicious_transactions.to_csv("suspicious_transactions.csv", index=False)

print("All output CSV files created successfully.")

# Final Project Summary

In this project, raw retail Black Friday transaction data was processed into business-ready analytical outputs.

## Work Completed

1. Loaded retail transaction CSV data
2. Inspected schema, nulls, duplicates, and data types
3. Cleaned duplicate records
4. Converted purchase dates into proper datetime format
5. Created date, discount, and order value features
6. Built category, city, payment, segment, hourly, and Black Friday reports
7. Created a KPI table
8. Created a customer summary table
9. Built RFM customer segmentation
10. Detected suspicious transactions using rule-based logic
11. Exported final CSV outputs

## Concepts Practiced

- Data ingestion
- Data cleaning
- Data transformation
- Feature engineering
- GroupBy aggregation
- Customer analytics
- RFM segmentation
- Rule-based anomaly detection
- Analytics-ready CSV export